# DigIA
## Pipeline de Visão Computacional e Testes de Generalização Extrema

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report
import time

#### Fase 1: Carregamento e Análise Exploratória de Imagens (EDA)

In [ ]:
# Carregando dataset MNIST a partir da OpenML
print("Baixando o dataset MNIST...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
print("Finalizou o processo de baixar o dataset MNIST!")

In [ ]:
# Separação entre features e target
X, y = mnist.data, mnist.target


In [ ]:
# Exploração da dimensionalidade das matrizes
print (f'========>> A matriz X possui {X.shape[0]} amostras, sendo {X.shape[1]} pixels por imagem')
print (f'========>> A matriz y é unidimensional e possui {y.shape[0]} rótulos / targets\n')
print (f'========>> Matriz de pixels de uma amostra:\n\n {X[10000]}')


In [ ]:
# Verificação da distribuição das classes e balanceamento do dataset
rotulos, freq_abs = np.unique(y, return_counts=True)
freq_perc = np.round(freq_abs / freq_abs.sum() * 100, decimals = 2)
print (f'Rótulos únicos: {rotulos}')
print (f'Frequências absolutas: {freq_abs}')
print (f'Frequências percentuais: {freq_perc}')

As classes presentes no dataset são 10 dígitos entre 0 e 9<br>
O dataset tem uma distribuição balanceada de um modo geral

In [ ]:
# Averiguação dos tipos de dados
print (f'Tipos de dados da matriz X: {X.dtype}')
print (f'Tipos de dados da matriz y: {y.dtype}')

A matriz de rótulos contém dados do tipo `object`, que no caso são strings. Vamos transformar estes dados para inteiros.

In [ ]:
# Alteração do tipo de dados da matriz de rótulos para inteiro
y = y.astype(np.uint8)

In [ ]:
# Geração de grade visual com 24 amostras aleatóras
random_idx = np.random.randint(0, X.shape[0], size=24)

fig, axes = plt.subplots(4, 6, figsize=[10, 7])
axes = axes.ravel()

for i, idx_i in enumerate(random_idx):
    xval = X[idx_i].reshape(28, 28)
    yval = int(y[idx_i])
    axes[i].imshow(xval, cmap='gray', vmin=0, vmax=255)
    axes[i].axis('off')
    axes[i].set_title(f"Dígito {yval}", fontsize=11, fontweight='bold')

plt.suptitle("24 Amostras Aleatórias do MNIST", fontsize=16)
plt.tight_layout()
plt.show()


Cada amostra do Dataset MNIST consiste de uma matriz unidimensional com tamanho 784 e são a representação achatada de uma figura com 28x28 pixels na qual cada um deles possui um valor que representa a intensidade de luz emitida por ele. Aqui, o canal de cor é único e portanto apenas um valor é o suficiente e pode variar de 0 (ausência de luz) a 255 (intensidade total de luz).<br>

Para representar as imagens na tela, é necessário rearranjar a matriz unidimensional e uma matriz bidimensional de dimensões 28x28. A função `reshape()`do Numpy é fundamental para isto.<br>

A plotagem utilizou uma representação baseada em tons de cinza (Grayscale), sendo que o fundo das figuras é preto e as regiões que contém os traços aproximam-se mais do branco. Comparando a figura com a matriz numérica observamos esta correspondência pois a maior parte dos pixels tem valor 0 (preto)

#### Fase 2: Pipeline de Pré-processamento e Divisão dos Dados

In [ ]:
# divisão estratificada dos dados em conjunto de treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify = y, random_state=42)

In [ ]:
# escalonamento dos dados
scaler = MinMaxScaler()

X_train_escalonado = scaler.fit_transform(X_train)
X_test_escalonado = scaler.transform(X_test)

Importância da normalização: o escalonador `MinMaxScaler` por padrão redimensiona os dados numéricos proporcionalmente para que resultem em valores entre 0 e 1. Modelos lineares e/ou que utilizam distâncias métricas podem ter oscilações muito grandes com valores com magnitudes diferentes e terão dificuldade de convergir matematicamente.<br>
Outro escalonador possível seria o `StandarScaler`, que redimensiona os dados para que tenha média zero e desvio padrão 1 (normalização). No caso do MNIST pode não ser o melhor caminho, pois as imagens do dataset tem todas um fundo preto (0), o que garante algum contraste para identificar o dígito. Estes pixels são predominantes. Com a normalização, estes pixels passariam a adotar valores muito baixos, mas ainda diferente de 0 e podem "borrar" as imagens e dificultar o treinamento.<br>

#### Fase 3: Implementação e Treinamento dos 3 Modelos

In [ ]:
def procura_melhores_parametros (nome_modelo, modelo, parametros, X_treino, y_treino):
    _, X_treino_gs, _, y_treino_gs = train_test_split(
    X_treino, y_treino, test_size=9000, stratify=y_treino, random_state=42
)
    # 3. Configura o Grid Search com Validação Cruzada de 3 folds
    grid_search = GridSearchCV(
        estimator=modelo,
        param_grid=parametros,
        cv=3,
        scoring="accuracy",
        n_jobs=-1,  # Usa todos os núcleos do processador em paralelo
        verbose=3,
    )

    # 4. Executa a busca
    print(f"Iniciando a busca de parâmetros otimizados do modelo {nome_modelo}...")
    inicio = time.time()
    grid_search.fit(X_treino_gs, y_treino_gs)
    fim = time.time()

    print(f"\nGrid Search concluído em {(fim - inicio)/60:.2f} minutos")

    print(f"Melhor combinação encontrada: {grid_search.best_params_}")
    print(f"Melhor Acurácia de Validação Cruzada: {grid_search.best_score_:.4f}")

    # Armazena o melhor classificador para o pipeline final
    return grid_search.best_estimator_

In [ ]:
def treina_modelo(nome_classificador, classificador, Xtreino, ytreino, Xteste, yteste):
    classificador = m['modelo']
    print(f'Iniciando o treinamento final do modelo {nome_classificador} com toda a base de treino')
    inicio_treino = time.time()
    classificador.fit(Xtreino, ytreino)
    fim_treino = time.time()
    tempo_treino = fim_treino - inicio_treino
    print(f"Treinamento concluído com sucesso em {tempo_treino/60:.2f} minutos!")

    print('Calculando predições nos dados de treino')
    y_pred_train = classificador.predict(Xtreino)
    print('Calculando predições nos dados de teste')
    y_pred_test = classificador.predict(Xteste)

    acuracia_treino = classification_report(ytreino,y_pred_train, output_dict=True, zero_division=0)['accuracy']
    acuracia_teste = classification_report(yteste,y_pred_test, output_dict=True, zero_division=0)['accuracy']
    print (f'Acurácia treino: {acuracia_treino}')
    print (f'Acurácia teste: {acuracia_teste}')
    relatorio_metricas = classification_report(yteste,y_pred_test, output_dict=True, zero_division=0)['weighted avg']
    print (f'Métricas do modelo: \n{relatorio_metricas}')
    matriz_confusao = confusion_matrix(yteste, y_pred_test)

    return {'nome_modelo':nome_classificador,'acuracia':acuracia_teste,
            'precision':relatorio_metricas['precision'],'recall':relatorio_metricas['recall'],
            'f1-score': relatorio_metricas['f1-score'], 'tempo_treino':tempo_treino}, matriz_confusao, classificador

In [ ]:
modelos_testar = [
    {
    'nome_modelo': 'SVM',
    'param_grid':{"kernel": ["linear", "rbf"],"C":[1,10]},
    'modelo_base': SVC(cache_size=1000, random_state=42)
    # 2. Instancia o SVC alocando mais memória RAM para o cálculo (cache_size=1000)
     }
    ]

melhores_modelos = []
for i in modelos_testar:
    nome_modelo = i['nome_modelo']
    param_grid = i['param_grid']
    modelo_base = i['modelo_base']

    melhor_modelo = procura_melhores_parametros(nome_modelo, modelo_base, param_grid, X_train_escalonado, y_train)
    melhores_modelos.append({'nome_modelo':nome_modelo, 'modelo': melhor_modelo})


#### Fase 4: Avaliação Comparativa de Desempenho

In [ ]:
def plota_matriz_confusao(matriz_confusao, nome_modelo):
 
    plt.figure(figsize=(10, 8))

    sns.heatmap(
        matriz_confusao,
        annot=True,  # Exibe os números dentro dos quadrados
        fmt="d",  # Formato de número inteiro (integuer)
        cmap="Blues",  # Gradiente de azul (quadrados escuros = muitos acertos)
        linewidths=0.5,  # Linha sutil separando os quadrados
        linecolor="silver",  # Cor da linha de separação
        cbar=True,  # Exibe a barra lateral de intensidade
        xticklabels=list(range(10)),  # Labels do eixo X (0 a 9)
        yticklabels=list(range(10)),  # Labels do eixo Y (0 a 9)
    )

    # 4. Configurações de títulos e eixos
    plt.title(
        f"Matriz de Confusão - Modelo {nome_modelo}\nProjeto DigIA",
        fontsize=14,
        pad=20,
        fontweight="bold",
    )
    plt.xlabel("Dígito Predito pelo Modelo", fontsize=12, labelpad=10)
    plt.ylabel("Dígito Real", fontsize=12, labelpad=10)

    # 5. Rotaciona os labels para melhor leitura
    plt.xticks(rotation=0)
    plt.yticks(rotation=0)

    # 6. Ajusta o layout para não cortar as bordas
    plt.tight_layout()

    # 7. Exibe o gráfico
    plt.show()

In [ ]:
resultados_modelos = []
matrizes = []
modelos_treinados = []

for m in melhores_modelos:

    nome_classificador = m['nome_modelo']
    classificador = m['modelo']
    
    resultados_modelo, matriz_confusao, classificador_treinado = treina_modelo(
        nome_classificador, classificador, X_train_escalonado, y_train, X_test_escalonado, y_test)
    
    resultados_modelos.append(resultados_modelo)
    matrizes.append({'classificador':nome_classificador, 'matriz_confusao': matriz_confusao})
    modelos_treinados.append({'modelo': nome_classificador, 'modelo_treinado': classificador_treinado})


In [ ]:
classificador_plotar = matrizes[0]['classificador']
matriz_plotar = matrizes[0]['matriz_confusao']

plota_matriz_confusao(matriz_confusao, classificador_plotar)

In [ ]:
pd.DataFrame(resultados_modelos)

"Dado que o dataset MNIST apresenta classes balanceadas e o custo de erro entre os dígitos é simétrico, a Acurácia apresenta-se como a métrica principal mais válida para avaliar o desempenho global dos modelos. Complementarmente, utiliza-se o F1-Score (Macro) para assegurar que a performance se mantém uniforme e robusta individualmente em cada uma das 10 classes, sem penalizações ocultas por falsos positivos ou falsos negativos em dígitos específicos."

#### Fase 5: Testes de generalização

##### Fase 5.1: Treinamento Restrito com Classes Ocultadas (Class Masking)

In [ ]:
# ocultar 3 e 6
filtro_ood_treino = (y_train != 3) & (y_train != 6)
filtro_ood_teste = (y_test == 3) | (y_test == 6)

X_train_ood = X_train[filtro_ood_treino]
y_train_ood = y_train[filtro_ood_treino]
X_test_ood = X_test[filtro_ood_teste]
y_test_ood = y_test[filtro_ood_teste]

print(f"--- Separando conjunto de treino com classes 3 e 6 ocultas ---")
print(f"Amostras no treino original: {X_train.shape}")
print(f"Amostras no treino restrito (sem 3 e 6): {X_train_ood.shape}")
print(f"Classes presentes no novo treino: {np.unique(y_train_ood)}")

print('Escalonando conjuntos de treino e teste')
scaler_ood = MinMaxScaler()
X_train_ood_escalonado = scaler_ood.fit_transform(X_train_ood)
X_test_ood_escalonado = scaler_ood.transform(X_test_ood)



#####  5.2 Teste de Generalização Extrema (Inferência OOD)

In [ ]:
m = melhores_modelos[0]
nome_classificador = m['nome_modelo']
classificador = m['modelo']
resultados_modelo, matriz_confusao, classificador_treinado = treina_modelo(
    nome_classificador, classificador, X_train_ood, y_train_ood, X_test_ood, y_test_ood)

In [ ]:
classificador_plotar = nome_classificador
matriz_plotar = matriz_confusao

plota_matriz_confusao(matriz_confusao, classificador_plotar)

Melhorias:
Tentar uso do StandardScaler ao invés do MinMaxScaler. É possível que a suavização das imagens para treino ajudem a reconhecer imagens reais.